In [2]:
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.linalg import eigh, sqrtm
import string
from qutip import Qobj, ptrace
from scipy.special import expit

def build_ssh_hamiltonian(N, lam, periodic=False, verbose=False):
    """
    Constructs the SSH Hamiltonian with alternating hoppings t1 = 1 - λ, t2 = 1 + λ.

    Parameters:
        N (int): Number of sites
        lam (float): SSH lambda parameter
        periodic (bool): Whether to add periodic boundary condition
        verbose (bool): If True, prints the resulting Hamiltonian

    Returns:conda init powershell

        ndarray: NxN Hamiltonian matrix
    """
    H = np.zeros((N, N), dtype=np.float64)
    t1, t2 = 1 - lam, 1 + lam

    max_i = N if periodic else N - 1
    for i in range(max_i):
        j = (i + 1) % N
        hopping = t1 if i % 2 == 0 else t2
        H[i, j] = H[j, i] = hopping

    if verbose:
        print(f"SSH Hamiltonian (N={N}, λ={lam}, periodic={periodic}):")
        print(H)

    return H

def compute_rho(beta, lam, pair_idx, N=4, periodic=False):
    """
    Computes the reduced 2-site density matrix (RDM) for a thermal state of the SSH model.
    For N=2, it directly constructs the 2x2 correlation matrix and embeds it in the 4x4 basis.

    Parameters:
        beta (float): Inverse temperature.
        lam (float): SSH lambda parameter.
        pair_idx (tuple): Indices (i, j) of the two sites to trace over.
        N (int): Number of sites in the full chain.
        periodic (bool): Whether to use periodic boundary conditions.

    Returns:
        rho (4x4 ndarray): The 2-site reduced density matrix.
    """

    if N == 2:
        # Build 2-site SSH Hamiltonian
        t1 = 1 - lam
        H_sp = np.array([[0, t1], [t1, 0]])
        eigvals, eigvecs = eigh(H_sp)

        # Fermi-Dirac distribution (no need for overflow handling here, since 2x2)
        f = 1 / (np.exp(beta * eigvals) + 1)
        f_diag = np.diag(f)

        # Correlation matrix (2x2)
        C = eigvecs @ f_diag @ eigvecs.T

        # Embed into 4x4 basis: |00>, |01>, |10>, |11>
        rho = np.zeros((4, 4), dtype=complex)
        rho[1, 1] = C[0, 0]
        rho[1, 2] = C[0, 1]
        rho[2, 1] = C[1, 0]
        rho[2, 2] = C[1, 1]

        return rho

    # General case for N > 2 using Peschel's method
    H = build_ssh_hamiltonian(N, lam, periodic)
    eigvals, eigvecs = eigh(H)

    # Overflow-safe Fermi-Dirac distribution
    if beta > 100:
        f_k = expit(-beta * eigvals)
    else:
        x = beta * eigvals
        x = np.clip(x, -700, 700)
        f_k = 1 / (np.exp(x) + 1)

    # Correlation matrix
    C = sum(np.outer(eigvecs[:, k], eigvecs[:, k]) * f_k[k] for k in range(N))

    # Extract 2x2 submatrix for given site pair
    i, j = pair_idx
    C_sub = C[[i, j]][:, [i, j]]
    evals_C, U = eigh(C_sub)
    evals_C = np.clip(evals_C, 1e-10, 1 - 1e-10)
    H_red = U @ np.diag(np.log((1 - evals_C) / evals_C)) @ U.T
    h11, h12 = H_red[0, 0], H_red[0, 1]
    h21, h22 = H_red[1, 0], H_red[1, 1]

    # Build reduced Hamiltonian in occupation basis
    H_occ = np.array([
        [0, 0, 0, 0],
        [0, h22, h12, 0],
        [0, h21, h11, 0],
        [0, 0, 0, h11 + h22]
    ])
    evals_occ, evecs_occ = eigh(H_occ)
    exp_neg_evals = np.exp(-evals_occ)
    Z = np.sum(exp_neg_evals)
    rho = evecs_occ @ np.diag(exp_neg_evals / Z) @ evecs_occ.T

    return rho

